# Contribution Guidelines

We welcome contributions! Please follow these guidelines so that your contribution can be reviewed and accepted easily.

## Table of Contents

1. [Getting Started](#getting-started)
2. [How to Contribute](#how-to-contribute)
3. [Running the Tests](#running-the-tests)
4. [Pre-commit Hooks and CI](#pre-commit-hooks-and-ci)
5. [Coding Standards](#coding-standards)
6. [Documentation](#documentation)
7. [Review Process](#review-process)
8. [Code of Conduct](#code-of-conduct)

## Getting Started

You need:

1. A GitHub account
2. Git
3. [UV](https://github.com/astral-sh/uv) (see [Installation](1_2_installation.ipynb))
4. Docker, only to run the database integration tests

## How to Contribute

### Fork and Clone the Repository

1. Click `Fork` on the [CIM-Graph repository](https://github.com/PNNL-CIM-Tools/CIM-Graph) to create a copy in your GitHub account.
2. Clone your fork and install the development environment with every database driver:
    ```sh
    git clone https://github.com/YOUR_USERNAME/CIM-Graph.git
    cd CIM-Graph
    uv sync --all-extras
    uv run pre-commit install
    ```

### Create a Branch

Branch from `develop`:
```sh
git checkout -b feature/your-feature-name develop
```

### Make Changes

1. Make your changes, following the [Coding Standards](#coding-standards).
2. Add or update tests, and check that they pass (see [Running the Tests](#running-the-tests)).
3. Describe user-visible changes in `CHANGELOG.md`.

### Commit Changes

```sh
git add <changed files>
git commit -S -m "Add feature: description of your feature"
```

**Note:** All commits must be signed. You can configure commit signing by following [these instructions](https://docs.github.com/en/authentication/managing-commit-signature-verification/signing-commits).

### Push to GitHub

```sh
git push origin feature/your-feature-name
```

### Submit a Pull Request

Open a pull request from your branch against `develop`, with a clear title and description.

## Running the Tests

Run the tests that need no database:

```sh
uv run pytest -m "not integration"
```

Tests that need a live database are marked `@pytest.mark.integration('<service>')`, where the service is `blazegraph`, `graphdb`, `neo4j`, `mysql` or `gridappsd`. They are skipped when nothing listens on the service's port (see `tests/conftest.py`). Mark new database tests the same way:

```python
@pytest.mark.integration('neo4j')
def test_neo4j_feeder():
    ...
```

`docker-compose.yml` runs Blazegraph, GraphDB and Neo4j with no stored data, and `tests/seed_databases.py` loads `tests/test_models/ieee13.xml` into each:

```sh
docker compose up -d --wait
uv run python tests/seed_databases.py
uv run pytest -m integration
docker compose down
```

`tests/test_backends_match_xml.py` checks that every object each database adapter reads matches what `XMLFile` reads. The GridAPPS-D tests need the GridAPPS-D platform (gridappsd-docker) instead, which has its own Blazegraph on port 8889, so stop the `blazegraph` service first. The MySQL tests need a `tier5_ieee13` database.

## Pre-commit Hooks and CI

The pre-commit hooks run on every commit once installed (`uv run pre-commit install`). To run them on all files:

```sh
uv run pre-commit run --all-files
```

They check YAML, JSON, TOML and XML files, fix trailing whitespace, file endings and double-quoted strings, sort imports with isort, run ruff for syntax errors and undefined names, and reject new files over 500 kB and commits to `main`.

CI (`.github/workflows/ci.yml`) runs on every pull request:

* the offline tests (`pytest -m "not integration"`) on Python 3.10 to 3.14;
* the offline tests with the lowest declared version of every direct dependency;
* the integration tests against the `docker-compose.yml` databases, seeded with IEEE 13;
* the ruff check for syntax errors and undefined names.

## Coding Standards

The rules for code in this repository are in [AGENTS.md](https://github.com/PNNL-CIM-Tools/CIM-Graph/blob/develop/AGENTS.md). In short: keep it simple, don't repeat logic, avoid abstractions and fallbacks that are not needed, fail fast with clear errors, and catch exceptions only where they can be handled. Also:

1. **Style**: Follow [PEP 8](https://peps.python.org/pep-0008/), with lines of up to 99 characters and single-quoted strings.
2. **Docstrings**: Use clear and descriptive docstrings for functions and classes.
3. **Type Annotations**: Annotate the types of function arguments and return values.
4. **Units**: Create physical quantities with their unit and convert them with `.to()` (`cim.ActivePower(5, 'MW')`, `load.p.to('kW')`). Never multiply by `1e3` or `1e6` by hand.
5. **Testing**: Write tests for your code, and make sure all tests pass before you submit a pull request.

## Documentation

The documentation is the Jupyter notebooks in `docs/`, built with MkDocs and mkdocs-jupyter. The notebooks are not executed when the site is built, so the saved outputs are what readers see: re-run a notebook after changing it, and leave no errors in its outputs. To preview the site:

```sh
pip install -r docs/requirements-mkdocs.txt
mkdocs serve
```

## Review Process

1. Once you submit a pull request, a maintainer reviews it.
2. Review comments may request changes to your code. Please stay open to feedback.
3. Make the changes and update the pull request. Make sure all commits are signed.
4. Once the pull request is approved and CI passes, it is merged into `develop`.

## Code of Conduct

We are committed to fostering a welcoming and inclusive community. By participating in this project, you agree to abide by our standards of professional engagement adopted from the [IEEE Code of Conduct](https://www.ieee.org/content/dam/ieee-org/ieee/web/org/about/ieee_code_of_conduct.pdf).

Thank you for your contributions!